# What Moved Voters Republican, 2016–2024: Cross-Pressure, Perceived Difference, and the Meaning of "Left"

Notebooks 11–15 established, election by election, that the voters who left the Democratic coalition held left economic views and conservative cultural ones. This notebook puts the pieces side by side and tests three claims that bear on strategy:

1. **Cross-pressure.** How large is the electorate that holds Democratic-aligned economic positions but votes Republican, in 2016 and 2024, and is cultural conservatism what holds them there?
2. **Perceived difference.** Did voters see an economic difference between Harris and Trump at all? The ANES asks respondents to place each candidate on the same 7-point scales on which they place themselves — government vs private health insurance, services, guaranteed jobs, abortion. If voters saw the candidates as far apart on abortion and close together on economics, the economic contrast was not communicated; if they saw the economic gap and voted on culture anyway, it was communicated and outweighed.
3. **What "left" means.** When a voter places themselves or the Democratic Party on a left-right scale, is that placement driven by their economic views or their cultural ones? And does media diet move where they place Harris, net of their own positions?

**Inputs:** `data/raw/anes/anes_2024_core.csv`; `data/raw/ces/CCES24_Common_OUTPUT_vv_topost_final.csv`; `data/raw/ces/CCES16_Common_OUTPUT_Feb2018_VV.tab`; CES 2024 harmonized `ces_2024.csv` (media items).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables')
def wmean(x, w):
    m = x.notna() & w.notna()
    return np.average(x[m], weights=w[m]) if m.any() else np.nan
def wshare(mask, w): return w[mask].sum() / w.sum()
def z(s): return (s - s.mean()) / s.std()
def wlogit(y, X, w):
    ok = X.notna().all(axis=1) & y.notna() & w.notna()
    y, X, w = y[ok], X[ok], w[ok]
    r = sm.GLM(y, sm.add_constant(X), family=sm.families.Binomial(), freq_weights=w / w.mean()).fit(cov_type='HC1')
    out = pd.DataFrame({'coef': r.params, 'se': r.bse, 'p': r.pvalues}).round(3)
    out.attrs['N'] = int(ok.sum()); print(f'  [N={int(ok.sum())}, events={int(y.sum())}]')
    return out

## 1. CES 2024 and CES 2016: the cross-pressured electorate

In [2]:
# --- CES 2024 (same items and indices as notebook 11) ---
cols = ['TS_g2024', 'vvweight_post', 'vvweight', 'presvote20post', 'CC24_410', 'pid7', 'birthyr', 'educ', 'race', 'hispanic',
        'CC24_321d', 'CC24_321e', 'CC24_323a', 'CC24_323b', 'CC24_323c', 'CC24_323d', 'CC24_323f', 'CC24_324a',
        'CC24_328b', 'CC24_328c', 'CC24_328d', 'CC24_328e', 'CC24_330a', 'CC24_330d', 'CC24_330e', 'CC24_340c', 'CC24_340f', 'CC24_341a', 'CC24_341b', 'CC24_341c',
        'CC24_300b_4', 'CC24_300b_5', 'CC24_300b_6']
d = pd.read_csv(RAW_DIR / 'ces' / 'CCES24_Common_OUTPUT_vv_topost_final.csv', usecols=cols, low_memory=False)
sup = lambda c: (d[c] == 1).astype(float).where(d[c].notna()); opp = lambda c: (d[c] == 2).astype(float).where(d[c].notna())
items = pd.DataFrame({
    'expand_medicaid': sup('CC24_328e'), 'keep_ACA': opp('CC24_328d'), 'no_medicaid_workreq': opp('CC24_328c'), 'student_loans': sup('CC24_323f'),
    'corp_tax_28': sup('CC24_341b'), 'tax_400k': sup('CC24_341c'), 'no_2017_cuts': opp('CC24_341a'), 'housing_subsidy': sup('CC24_328b'),
    'no_wall': opp('CC24_323c'), 'allow_asylum': opp('CC24_340f'), 'no_more_border_patrol': opp('CC24_323b'), 'legal_status': sup('CC24_323a'), 'dreamers': sup('CC24_323d'),
    'abortion_choice': sup('CC24_324a'), 'samesex_marriage': sup('CC24_340c'), 'no_more_police': opp('CC24_321d'), 'defund': sup('CC24_321e')})
ECON = ['expand_medicaid', 'keep_ACA', 'no_medicaid_workreq', 'student_loans', 'corp_tax_28', 'tax_400k', 'no_2017_cuts', 'housing_subsidy']
IMMIG = ['no_wall', 'allow_asylum', 'no_more_border_patrol', 'legal_status', 'dreamers']; SOCIAL = ['abortion_choice', 'samesex_marriage']; POLICE = ['no_more_police', 'defund']
d['econ_left'] = items[ECON].mean(axis=1); d['immig_cons'] = 1 - items[IMMIG].mean(axis=1); d['social_cons'] = 1 - items[SOCIAL].mean(axis=1)
d['police_cons'] = 1 - items[POLICE].mean(axis=1); d['cult_cons'] = 1 - items[IMMIG + SOCIAL + POLICE].mean(axis=1)
d['voted24'] = d['TS_g2024'].notna() & (d['TS_g2024'] != 7)
d['v24'] = d['CC24_410'].map({1: 'Harris', 2: 'Trump'})
d['v20'] = d['presvote20post'].map({1: 'Biden', 2: 'Trump', 6: 'nonvoter'})
c24 = d[d['voted24'] & d['v24'].notna()].copy(); c24['wt'] = c24['vvweight_post'].fillna(c24['vvweight'])
c24['trump'] = (c24['v24'] == 'Trump').astype(float)
for c in ['CC24_330a', 'CC24_330d', 'CC24_330e']: c24[c] = c24[c].where(c24[c] <= 7)
c24['fox'] = (c24['CC24_300b_5'] == 1).astype(float); c24['msnbc'] = (c24['CC24_300b_6'] == 1).astype(float); c24['cnn'] = (c24['CC24_300b_4'] == 1).astype(float)
c24['pid7c'] = c24['pid7'].where(c24['pid7'] <= 7)
print('CES 2024 validated voters with a major-party vote:', len(c24))
# cross-pressure: economic-left (index >= 0.5) by vote
c24['econ_left_bin'] = c24['econ_left'] >= 0.5
c24['cult_cons_bin'] = c24['cult_cons'] >= 0.5
xp = c24.groupby(['econ_left_bin', 'cult_cons_bin']).apply(lambda s: pd.Series({'share_of_electorate': s.wt.sum() / c24.wt.sum(), 'trump_share': wmean(s.trump, s.wt), 'N': len(s)}))
print('\n2024 electorate by economic-left x culturally-conservative (index >= 0.5):'); print(xp.round(3))
tv = c24[c24.trump == 1]
print('\nTrump 2024 voters who are economically left (index >= 0.5): %.1f%%; of those, culturally conservative: %.1f%%' % (
    100*wshare(tv.econ_left_bin, tv.wt), 100*wshare(tv[tv.econ_left_bin].cult_cons_bin, tv[tv.econ_left_bin].wt)))
print('Economically-left voters as share of the 2024 electorate: %.1f%%; their Trump share: %.1f%%' % (100*wshare(c24.econ_left_bin, c24.wt), 100*wmean(c24[c24.econ_left_bin].trump, c24[c24.econ_left_bin].wt)))
xp.to_csv(TABLES_DIR / 'crosspressure_2024.csv')
# Cutoff sensitivity: >= 0.5 counts an exact tie (4 of 8 economic items, 4.5 of 9 cultural) as left / conservative
sens = []
for lab, el, cc in [('>= 0.5 (ties in)', c24.econ_left >= 0.5, c24.cult_cons >= 0.5), ('> 0.5 (ties out)', c24.econ_left > 0.5, c24.cult_cons > 0.5)]:
    x = c24.assign(e=el, k=cc).groupby(['e', 'k']).apply(lambda s: pd.Series({'share': s.wt.sum() / c24.wt.sum(), 'trump': wmean(s.trump, s.wt)}))
    sens.append(x.assign(cutoff=lab))
sens = pd.concat(sens); print('\nCutoff sensitivity:'); print(sens.round(3))
sens.to_csv(TABLES_DIR / 'crosspressure_2024_cutoff_sensitivity.csv')

CES 2024 validated voters with a major-party vote: 31834

2024 electorate by economic-left x culturally-conservative (index >= 0.5):
                             share_of_electorate  trump_share        N
econ_left_bin cult_cons_bin                                           
False         False                        0.011        0.443    284.0
              True                         0.356        0.988   9994.0
True          False                        0.433        0.038  15715.0
              True                         0.200        0.676   5841.0

Trump 2024 voters who are economically left (index >= 0.5): 29.8%; of those, culturally conservative: 89.2%
Economically-left voters as share of the 2024 electorate: 63.3%; their Trump share: 24.0%

Cutoff sensitivity:
             share  trump            cutoff
e     k                                    
False False  0.011  0.443  >= 0.5 (ties in)
      True   0.356  0.988  >= 0.5 (ties in)
True  False  0.433  0.038  >= 0.5 (ties in)
 

In [3]:
# Trump vote among ALL 2024 voters: economic vs cultural, standardized; and among the economically-left only
X = c24[['econ_left', 'immig_cons', 'social_cons', 'police_cons']].apply(z)
full = wlogit(c24.trump, X, c24.wt); print('All 2024 voters (N=%d):' % len(c24)); print(full)
el = c24[c24.econ_left_bin]
Xe = el[['immig_cons', 'social_cons', 'police_cons']].apply(z)
print('\nAmong economically-left voters only (N=%d): what predicts a Trump vote' % len(el)); print(wlogit(el.trump, Xe, el.wt))
# Item-level: Trump share among voters holding each Democratic-aligned economic position
rows = []
for it in ECON + IMMIG + SOCIAL + POLICE:
    s = c24[items.loc[c24.index, it] == 1]
    rows.append({'item': it, 'holders_share_of_electorate': s.wt.sum() / c24.wt.sum(), 'trump_share_among_holders': wmean(s.trump, s.wt)})
it24 = pd.DataFrame(rows).set_index('item'); print('\nTrump share among voters holding each Democratic-aligned position, 2024:'); print(it24.round(3))
it24.to_csv(TABLES_DIR / 'trump_share_by_dem_position_2024.csv')

  [N=31832, events=13695]
All 2024 voters (N=31834):
              coef     se    p
const       -0.287  0.028  0.0
econ_left   -2.248  0.046  0.0
immig_cons   1.781  0.042  0.0
social_cons  1.108  0.030  0.0
police_cons  0.227  0.031  0.0

Among economically-left voters only (N=21556): what predicts a Trump vote
  [N=21555, events=3742]
              coef     se    p
const       -2.453  0.035  0.0
immig_cons   1.607  0.032  0.0
social_cons  0.965  0.022  0.0
police_cons  0.383  0.029  0.0

Trump share among voters holding each Democratic-aligned position, 2024:
                       holders_share_of_electorate  trump_share_among_holders
item                                                                         
expand_medicaid                              0.758                      0.384
keep_ACA                                     0.602                      0.231
no_medicaid_workreq                          0.455                      0.332
student_loans                             

In [4]:
# --- CES 2016 (items as notebook 12) ---
cols16 = ['commonweight_vv_post', 'commonweight_vv', 'commonweight', 'CC16_326', 'CC16_410a', 'CC16_331_1', 'CC16_331_2', 'CC16_331_3', 'CC16_331_7', 'CC16_332a', 'CC16_351I', 'CC16_351K', 'pid7']
d16 = pd.read_csv(RAW_DIR / 'ces' / 'CCES16_Common_OUTPUT_Feb2018_VV.tab', sep='\t', usecols=cols16, low_memory=False)
d16['CC16_410a'] = pd.to_numeric(d16['CC16_410a'], errors='coerce')
d16['wt'] = d16['commonweight_vv_post'].fillna(d16['commonweight_vv']).fillna(d16['commonweight'])
d16['v16'] = d16['CC16_410a'].map({1: 'Trump', 2: 'Clinton'}); d16['v12'] = d16['CC16_326'].map({1: 'Obama', 2: 'Romney'})
sel = lambda c: (d16[c] == 1).astype(float).where(d16[c].notna())
d16['immig_cons'] = pd.concat([1 - sel('CC16_331_1'), sel('CC16_331_2'), 1 - sel('CC16_331_3'), sel('CC16_331_7')], axis=1).mean(axis=1)
d16['social_cons'] = (d16['CC16_332a'] == 2).astype(float).where(d16['CC16_332a'].notna())
d16['econ_left'] = pd.concat([1 - (d16['CC16_351I'] == 1).astype(float).where(d16['CC16_351I'].notna()), (d16['CC16_351K'] == 1).astype(float).where(d16['CC16_351K'].notna())], axis=1).mean(axis=1)
c16 = d16[d16.v16.notna()].copy(); c16['trump'] = (c16.v16 == 'Trump').astype(float)
c16['econ_left_bin'] = c16.econ_left >= 0.5; c16['cult_cons_bin'] = ((c16.immig_cons + c16.social_cons) / 2) >= 0.5
xp16 = c16.groupby(['econ_left_bin', 'cult_cons_bin']).apply(lambda s: pd.Series({'share_of_electorate': s.wt.sum() / c16.wt.sum(), 'trump_share': wmean(s.trump, s.wt), 'N': len(s)}))
print('2016 electorate by economic-left (keep ACA, $12 min wage) x culturally-conservative (immigration + abortion):'); print(xp16.round(3))
tv16 = c16[c16.trump == 1]
print('\nTrump 2016 voters economically left: %.1f%%; of those, culturally conservative: %.1f%%' % (100*wshare(tv16.econ_left_bin, tv16.wt), 100*wshare(tv16[tv16.econ_left_bin].cult_cons_bin, tv16[tv16.econ_left_bin].wt)))
print(wlogit(c16.trump, c16[['econ_left', 'immig_cons', 'social_cons']].apply(z), c16.wt))
xp16.to_csv(TABLES_DIR / 'crosspressure_2016.csv')
x = c16.assign(e=c16.econ_left > 0.5, k=((c16.immig_cons + c16.social_cons) / 2) > 0.5).groupby(['e', 'k']).apply(lambda s: pd.Series({'share': s.wt.sum() / c16.wt.sum(), 'trump': wmean(s.trump, s.wt)}))
print('2016 with strict > 0.5 (requires both economic items; ties out):'); print(x.round(3).unstack('k'))
print('\nNote: the 2016 economic index is two items and the 2024 index eight; shares are not comparable in level across years, only the pattern.')

2016 electorate by economic-left (keep ACA, $12 min wage) x culturally-conservative (immigration + abortion):
                             share_of_electorate  trump_share        N
econ_left_bin cult_cons_bin                                           
False         False                        0.052        0.761   2009.0
              True                         0.237        0.964   9478.0
True          False                        0.452        0.150  20018.0
              True                         0.259        0.581   9385.0

Trump 2016 voters economically left: 44.9%; of those, culturally conservative: 68.9%
  [N=40850, events=18732]
              coef     se    p
const       -0.280  0.016  0.0
econ_left   -1.591  0.020  0.0
immig_cons   1.218  0.019  0.0
social_cons  0.625  0.016  0.0
2016 with strict > 0.5 (requires both economic items; ties out):
       share         trump       
k      False  True   False  True 
e                                
False  0.275  0.335  0.556  0.

## 2. Did voters see an economic difference between Harris and Trump?

ANES respondents place themselves, Harris, and Trump on the same 7-point scales. For each scale: the share who placed Harris to the left of Trump (the correct direction), the share who placed them at the same point or reversed, and the share who could not place one or both. Then the perceived Harris–Trump distance on the three economic scales against the abortion scale, by vote group.

In [5]:
a = pd.read_csv(RAW_DIR / 'anes' / 'anes_2024_core.csv')
a = a[a.weight_post.notna() & (a.weight_post > 0)].copy(); a['wt'] = a.weight_post
a['group'] = np.select(
    [(a.vote20 == 'D') & (a.vote24 == 'D'), (a.vote20 == 'D') & (a.vote24 == 'R'), (a.vote20 == 'NV') & (a.vote24 == 'D'), (a.vote20 == 'NV') & (a.vote24 == 'R'),
     (a.vote20 == 'R') & (a.vote24 == 'R'), (a.vote20 == 'D') & (a.vote24 == 'NV')],
    ['Biden->Harris', 'Biden->Trump', 'New->Harris', 'New->Trump', 'Trump->Trump', 'Biden->stay home'], None)
ORDER = ['Biden->Harris', 'Biden->stay home', 'Biden->Trump', 'New->Harris', 'New->Trump', 'Trump->Trump']
# On every scale 1 = the left/liberal pole except services (1 = fewer services). Flip services so 1 = left.
for who in ['harris', 'trump']:
    a[f'{who}_services_l'] = 8 - a[f'{who}_services']
a['scale_services_l'] = 8 - a['scale_services']
SCALES = {'health insurance (govt vs private)': 'health_ins', 'services & spending': 'services_l', 'guaranteed jobs/income': 'jobs_income', 'abortion': 'abortion'}
rows = []
for lab, s in SCALES.items():
    h, t = a[f'harris_{s}'], a[f'trump_{s}']
    both = h.notna() & t.notna()
    rows.append({'scale': lab, 'placed both (%)': 100*wshare(both, a.wt), 'Harris left of Trump (% of all)': 100*wshare(both & (h < t), a.wt),
                 'same point (% of all)': 100*wshare(both & (h == t), a.wt), 'reversed (% of all)': 100*wshare(both & (h > t), a.wt),
                 'mean gap Trump-Harris (placers)': wmean((t - h)[both], a.wt[both])})
pc = pd.DataFrame(rows).set_index('scale'); print('All respondents with post weight (N=%d):' % len(a)); print(pc.round(1))
pc.to_csv(TABLES_DIR / 'anes_perceived_candidate_gap.csv')

All respondents with post weight (N=4964):
                                    placed both (%)  \
scale                                                 
health insurance (govt vs private)             91.8   
services & spending                            95.8   
guaranteed jobs/income                         92.3   
abortion                                       93.0   

                                    Harris left of Trump (% of all)  \
scale                                                                 
health insurance (govt vs private)                             71.5   
services & spending                                            70.9   
guaranteed jobs/income                                         72.6   
abortion                                                       77.6   

                                    same point (% of all)  \
scale                                                       
health insurance (govt vs private)                   12.3   
services & spend

In [6]:
# By vote group: perceived Trump-minus-Harris gap on each scale (positive = Harris seen to the left), and share unable to place
rows = []
for g in ORDER:
    s = a[a.group == g]; r = {'group': g, 'N': len(s)}
    for lab, sc in SCALES.items():
        h, t = s[f'harris_{sc}'], s[f'trump_{sc}']; both = h.notna() & t.notna()
        r[f'gap: {lab}'] = wmean((t - h)[both], s.wt[both]); r[f'cannot place: {lab}'] = 100*wshare(~both, s.wt)
    rows.append(r)
gg = pd.DataFrame(rows).set_index('group')
print('Perceived Trump minus Harris placement (7-pt; positive = Harris to the left):'); print(gg[[c for c in gg.columns if c.startswith('gap')]].round(2))
print('\nShare unable to place one or both candidates (%):'); print(gg[[c for c in gg.columns if c.startswith('cannot')] + ['N']].round(1))
gg.to_csv(TABLES_DIR / 'anes_perceived_gap_by_group.csv')
# economic vs abortion gap, and where switchers put Harris on health insurance
a['econ_gap'] = pd.concat([(a.trump_health_ins - a.harris_health_ins), (a.trump_services_l - a.harris_services_l), (a.trump_jobs_income - a.harris_jobs_income)], axis=1).mean(axis=1)
a['abortion_gap'] = a.trump_abortion - a.harris_abortion
for g in ORDER:
    s = a[a.group == g]
    print(f'{g:16s} mean econ gap {wmean(s.econ_gap, s.wt):+.2f}  abortion gap {wmean(s.abortion_gap, s.wt):+.2f}   saw Harris on govt-insurance side (<=3): {100*wshare(s.harris_health_ins <= 3, s.wt):.0f}%  saw Trump on private side (>=5): {100*wshare(s.trump_health_ins >= 5, s.wt):.0f}%')

Perceived Trump minus Harris placement (7-pt; positive = Harris to the left):
                  gap: health insurance (govt vs private)  \
group                                                       
Biden->Harris                                        3.24   
Biden->stay home                                     2.17   
Biden->Trump                                         1.81   
New->Harris                                          2.53   
New->Trump                                           2.36   
Trump->Trump                                         3.27   

                  gap: services & spending  gap: guaranteed jobs/income  \
group                                                                     
Biden->Harris                         3.64                         3.36   
Biden->stay home                      2.05                         2.64   
Biden->Trump                          1.84                         1.38   
New->Harris                           2.64                

In [7]:
# Does perceived economic difference predict the vote among Biden-2020 voters, net of own positions?
def sc(x, lo, hi): return (x - lo) / (hi - lo)
a['econ_left'] = pd.concat([sc(a.scale_services, 1, 7), sc(7 - a.scale_health_ins, 0, 6), sc(7 - a.scale_jobs_income, 0, 6), (a.min_wage == 1).astype(float).where(a.min_wage.notna()), sc(7 - a.paid_leave, 0, 6)], axis=1).mean(axis=1)
a['immig_cons'] = pd.concat([sc(7 - a.wall, 0, 6), sc(7 - a.dreamers, 0, 6), sc(7 - a.birthright_end, 0, 6), sc(a.immig_levels, 1, 5), sc(7 - a.immig_crime, 0, 6), sc(a.path_citizenship, 1, 7)], axis=1).mean(axis=1)
a['culture_cons'] = pd.concat([sc(a.trans_bathroom, 1, 7), sc(7 - a.trans_sports_ban, 0, 6), sc(a.trans_military, 1, 7), a.dei_campus.map({1: 0.0, 2: 1.0, 3: 0.5}), sc(a.diversity_better, 1, 5), sc(a.gay_marriage, 1, 7), sc(4 - a.abortion_std, 0, 3)], axis=1).mean(axis=1)
b = a[(a.vote20 == 'D') & a.vote24.isin(['D', 'R'])].copy(); b['switch'] = (b.vote24 == 'R').astype(float)
# closeness to each candidate on the economic scales (own position minus candidate placement)
nine = ['scale_health_ins', 'scale_services_l', 'scale_jobs_income', 'harris_health_ins', 'harris_services_l', 'harris_jobs_income', 'trump_health_ins', 'trump_services_l', 'trump_jobs_income']
allp = b[nine].notna().all(axis=1)   # require self and both candidates placed on all three scales
b['own_econ'] = b[nine[:3]].mean(axis=1).where(allp); b['harris_econ'] = b[nine[3:6]].mean(axis=1).where(allp); b['trump_econ'] = b[nine[6:]].mean(axis=1).where(allp)
b['closer_to_trump_econ'] = ((b.own_econ - b.trump_econ).abs() < (b.own_econ - b.harris_econ).abs()).astype(float).where(allp)
b['equidistant_econ'] = ((b.own_econ - b.trump_econ).abs() == (b.own_econ - b.harris_econ).abs()).astype(float).where(allp)
sub = b.dropna(subset=['econ_gap', 'abortion_gap', 'econ_left', 'immig_cons', 'culture_cons'])
X = sub[['econ_gap', 'abortion_gap', 'econ_left', 'immig_cons', 'culture_cons']].apply(z)
print('Switching among Biden-2020 voters on perceived candidate gaps + own positions (N=%d, events=%d):' % (len(sub), int(sub.switch.sum()))); print(wlogit(sub.switch, X, sub.wt))
for g, s in [('Biden->Harris', b[b.switch == 0]), ('Biden->Trump', b[b.switch == 1])]:
    s2 = s.dropna(subset=['closer_to_trump_econ'])
    print(f'{g}: of those placing self and both candidates on all three economic scales (N={len(s2)}): closer to Trump {100*wmean(s2.closer_to_trump_econ, s2.wt):.0f}%, equidistant {100*wmean(s2.equidistant_econ, s2.wt):.0f}%, closer to Harris {100*(1 - wmean(s2.closer_to_trump_econ, s2.wt) - wmean(s2.equidistant_econ, s2.wt)):.0f}%')
    print('   mean placements (health ins / services / jobs; 1 = left): self %.1f %.1f %.1f | Harris %.1f %.1f %.1f | Trump %.1f %.1f %.1f' % tuple(wmean(s[c], s.wt) for c in nine))

Switching among Biden-2020 voters on perceived candidate gaps + own positions (N=1922, events=63):
  [N=1922, events=63]
               coef     se      p
const        -4.543  0.234  0.000
econ_gap     -0.422  0.136  0.002
abortion_gap -0.119  0.106  0.258
econ_left    -0.293  0.121  0.016
immig_cons    0.505  0.132  0.000
culture_cons  0.959  0.178  0.000
Biden->Harris: of those placing self and both candidates on all three economic scales (N=1555): closer to Trump 7%, equidistant 1%, closer to Harris 92%
   mean placements (health ins / services / jobs; 1 = left): self 2.6 2.5 3.1 | Harris 3.0 2.5 3.0 | Trump 6.2 6.1 6.3
Biden->Trump: of those placing self and both candidates on all three economic scales (N=46): closer to Trump 67%, equidistant 10%, closer to Harris 23%
   mean placements (health ins / services / jobs; 1 = left): self 3.7 3.7 4.2 | Harris 3.1 2.7 3.2 | Trump 4.9 4.4 4.6


## 3. What "left" and "liberal" mean to voters

Regress self-placement on the respondent's own economic, immigration, and cultural indices. If cultural views explain most of where people put themselves, then a party seen as "too far left" is being judged on culture. Then regress where the respondent places the Democratic Party (ANES) and Harris (CES) on the respondent's positions and, in the CES, on media diet.

In [8]:
# ANES: self left-right (0-10) and Democratic Party placement
for dep, lab in [('lr_self', 'self placement (0 left - 10 right)'), ('lr_dem_party', 'Democratic Party placement'), ('lr_rep_party', 'Republican Party placement')]:
    sub = a.dropna(subset=[dep, 'econ_left', 'immig_cons', 'culture_cons'])
    X = sm.add_constant(sub[['econ_left', 'immig_cons', 'culture_cons']].apply(z))
    r = sm.WLS(sub[dep], X, weights=sub.wt).fit(cov_type='HC1')
    print(f'\nANES {lab} (N={len(sub)}, R2={r.rsquared:.3f}):'); print(pd.DataFrame({'coef': r.params, 'se': r.bse}).round(3))
# Among Biden-2020 voters only: what makes them place the Democrats further left than themselves?
sub = b.dropna(subset=['lr_self', 'lr_dem_party', 'econ_left', 'immig_cons', 'culture_cons']).copy(); sub['dem_left_of_self'] = sub.lr_self - sub.lr_dem_party
X = sm.add_constant(sub[['econ_left', 'immig_cons', 'culture_cons']].apply(z))
r = sm.WLS(sub.dem_left_of_self, X, weights=sub.wt).fit(cov_type='HC1')
print(f'\nBiden-2020 voters: (self minus Democratic Party) left-right distance on own positions (N={len(sub)}, R2={r.rsquared:.3f}):'); print(pd.DataFrame({'coef': r.params, 'se': r.bse}).round(3))


ANES self placement (0 left - 10 right) (N=4644, R2=0.426):
               coef     se
const         5.612  0.044
econ_left    -0.162  0.065
immig_cons    0.714  0.070
culture_cons  1.135  0.074

ANES Democratic Party placement (N=4625, R2=0.088):
               coef     se
const         2.914  0.057
econ_left     0.543  0.081
immig_cons   -0.062  0.091
culture_cons -0.333  0.092

ANES Republican Party placement (N=4636, R2=0.018):
               coef     se
const         7.504  0.058
econ_left    -0.446  0.087
immig_cons   -0.116  0.091
culture_cons -0.345  0.095

Biden-2020 voters: (self minus Democratic Party) left-right distance on own positions (N=1912, R2=0.122):
               coef     se
const         0.297  0.082
econ_left    -0.439  0.108
immig_cons   -0.083  0.133
culture_cons  0.653  0.105


In [9]:
# CES 2024: self-ideology (1 very liberal .. 7 very conservative) and Harris placement on own positions and media diet
c24['self_ideo'] = c24['CC24_330a']; c24['harris_ideo'] = c24['CC24_330d']
for dep, lab in [('self_ideo', 'self ideology'), ('harris_ideo', 'Harris placement')]:
    sub = c24.dropna(subset=[dep, 'econ_left', 'immig_cons', 'social_cons', 'police_cons'])
    X = sm.add_constant(sub[['econ_left', 'immig_cons', 'social_cons', 'police_cons']].apply(z))
    r = sm.WLS(sub[dep], X, weights=sub.wt).fit(cov_type='HC1')
    print(f'\nCES 2024 {lab}, all voters (N={len(sub)}, R2={r.rsquared:.3f}):'); print(pd.DataFrame({'coef': r.params, 'se': r.bse}).round(3))
# Harris placement among Biden-2020 voters: own positions + party ID + media
bb = c24[c24.v20 == 'Biden'].dropna(subset=['harris_ideo', 'econ_left', 'immig_cons', 'social_cons', 'police_cons', 'pid7c']).copy()
bb['harris_very_lib'] = (bb.harris_ideo == 1).astype(float)
X = bb[['econ_left', 'immig_cons', 'social_cons', 'police_cons']].apply(z); X['pid7c'] = bb.pid7c; X[['fox', 'msnbc', 'cnn']] = bb[['fox', 'msnbc', 'cnn']]
r = sm.WLS(bb.harris_ideo, sm.add_constant(X), weights=bb.wt).fit(cov_type='HC1')
print(f'\nBiden-2020 voters: Harris placement (1 very liberal .. 7) on positions, party ID, and TV news diet (N={len(bb)}, R2={r.rsquared:.3f}):'); print(pd.DataFrame({'coef': r.params, 'se': r.bse, 'p': r.pvalues}).round(3))
print('\nShare seeing Harris as "very liberal" by Fox viewing, Biden-2020 voters: Fox %.1f%% vs not %.1f%%' % (100*wmean(bb[bb.fox == 1].harris_very_lib, bb[bb.fox == 1].wt), 100*wmean(bb[bb.fox == 0].harris_very_lib, bb[bb.fox == 0].wt)))
print('...and within immigration-conservative Biden voters (index >= 0.5): Fox %.1f%% vs not %.1f%%' % (
    100*wmean(bb[(bb.fox == 1) & (bb.immig_cons >= 0.5)].harris_very_lib, bb[(bb.fox == 1) & (bb.immig_cons >= 0.5)].wt), 100*wmean(bb[(bb.fox == 0) & (bb.immig_cons >= 0.5)].harris_very_lib, bb[(bb.fox == 0) & (bb.immig_cons >= 0.5)].wt)))
# does Fox predict switching net of positions and perception?
bb['switch'] = (bb.v24 == 'Trump').astype(float)
X2 = X.copy(); X2['harris_ideo'] = bb.harris_ideo
print('\nSwitching among Biden-2020 voters: positions + party ID + media + Harris placement (N=%d, events=%d):' % (len(bb), int(bb.switch.sum()))); print(wlogit(bb.switch, X2, bb.wt))


CES 2024 self ideology, all voters (N=31366, R2=0.693):
              coef     se
const        4.011  0.011
econ_left   -0.586  0.018
immig_cons   0.479  0.021
social_cons  0.557  0.016
police_cons  0.282  0.016

CES 2024 Harris placement, all voters (N=30818, R2=0.409):
              coef     se
const        2.055  0.011
econ_left    0.345  0.020
immig_cons  -0.285  0.024
social_cons -0.144  0.017
police_cons -0.114  0.017

Biden-2020 voters: Harris placement (1 very liberal .. 7) on positions, party ID, and TV news diet (N=16251, R2=0.036):
              coef     se      p
const        2.795  0.046  0.000
econ_left    0.062  0.028  0.024
immig_cons  -0.109  0.025  0.000
social_cons  0.040  0.029  0.174
police_cons -0.082  0.021  0.000
pid7c       -0.005  0.019  0.807
fox         -0.140  0.101  0.168
msnbc       -0.032  0.040  0.417
cnn         -0.009  0.047  0.849

Share seeing Harris as "very liberal" by Fox viewing, Biden-2020 voters: Fox 23.8% vs not 8.6%
...and within immigratio

## 4. Summary table

In [10]:
summ = pd.DataFrame({
    'econ-left share of electorate': [100*wshare(c16.econ_left_bin, c16.wt), 100*wshare(c24.econ_left_bin, c24.wt)],
    'Trump share among econ-left': [100*wmean(c16[c16.econ_left_bin].trump, c16[c16.econ_left_bin].wt), 100*wmean(c24[c24.econ_left_bin].trump, c24[c24.econ_left_bin].wt)],
    'Trump voters who are econ-left': [100*wshare(tv16.econ_left_bin, tv16.wt), 100*wshare(tv.econ_left_bin, tv.wt)],
    'econ-left & cult-cons: Trump share': [100*xp16.loc[(True, True), 'trump_share'], 100*xp.loc[(True, True), 'trump_share']],
    'econ-left & cult-lib: Trump share': [100*xp16.loc[(True, False), 'trump_share'], 100*xp.loc[(True, False), 'trump_share']]}, index=['2016', '2024'])
print(summ.round(1)); summ.to_csv(TABLES_DIR / 'crosspressure_summary.csv')
print('\nSaved 7 tables')

      econ-left share of electorate  Trump share among econ-left  \
2016                           71.1                         30.8   
2024                           63.3                         24.0   

      Trump voters who are econ-left  econ-left & cult-cons: Trump share  \
2016                            44.9                                58.1   
2024                            29.8                                67.6   

      econ-left & cult-lib: Trump share  
2016                               15.0  
2024                                3.8  

Saved 7 tables


## Limitations

- **Index composition differs by year.** 2016 economics is two items (keep ACA, $12 minimum wage); 2024 is eight. The 2016 "economically left" share is therefore not comparable in level with 2024; the pattern within each year is.
- **Candidate placements are a knowledge test as much as a perception test.** "Cannot place" mixes not knowing the candidate's position with not having a position oneself (the 99 "haven't thought" code applies to both). The perceived gap is computed only among those who placed both.
- **Media diet is self-reported network viewing in the past 24 hours**, not exposure to any particular story. The Fox coefficient is a diet association, not an effect of coverage; a Biden voter who watches Fox in 2024 already differs from one who does not.
- **Left-right and liberal-conservative self-placement are single items** that many respondents use as partisan labels. The regressions say what covaries with the label, not what the respondent means by it.